# CNN nhận dạng chữ số MNIST: giải thích từ đầu

Notebook này xây dựng một CNN nhỏ bằng TensorFlow/Keras để phân loại ảnh chữ số viết tay từ **0 đến 9**. Nội dung được sắp xếp theo luồng: hiểu dữ liệu → chuẩn bị đầu vào/nhãn → xây mô hình → huấn luyện → đánh giá → dự đoán một ảnh.

> **Lưu ý về “from scratch”:** ở đây cụm từ này có nghĩa là tự khai báo kiến trúc và quy trình huấn luyện của một CNN cơ bản. Các phép tích chập, tính gradient và cập nhật trọng số vẫn do TensorFlow/Keras thực hiện; notebook **không** tự viết lại các phép toán đó.

Nếu môi trường chưa có TensorFlow, hãy cài đặt `tensorflow` trong đúng môi trường Python đang dùng cho notebook, sau đó chạy lại kernel.

## 1. Mục tiêu và sơ đồ tổng thể

Sau notebook, bạn sẽ hiểu vì sao ảnh cần chuẩn hóa và thêm chiều kênh, cách đầu ra 10 lớp được biểu diễn, và cách kích thước tensor thay đổi qua từng lớp CNN.

**Luồng xử lý:**
1. Tải MNIST, xem kích thước, nhãn và một vài ảnh.
2. Chuẩn hóa pixel, thêm chiều kênh và tách tập validation từ tập train.
3. Xây CNN: tích chập → ReLU → pooling → flatten → dense → softmax.
4. Huấn luyện với categorical cross-entropy, Adam và EarlyStopping.
5. Đánh giá trên test, xem báo cáo từng lớp và thử dự đoán ảnh.

## 2. Import thư viện và cố định seed

TensorFlow cung cấp dữ liệu MNIST và các lớp CNN. NumPy dùng để thao tác mảng; Matplotlib để vẽ ảnh/đồ thị; scikit-learn để tạo tập validation và tính các chỉ số phân loại. Cố định seed giúp các thao tác ngẫu nhiên có tính tái lập tốt hơn, dù kết quả vẫn có thể khác giữa phần cứng hoặc phiên bản thư viện.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.utils import to_categorical

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)

## 3. Tải dữ liệu MNIST

`mnist.load_data()` trả về hai cặp `(ảnh, nhãn)`: train có 60.000 mẫu và test có 10.000 mẫu. Mỗi ảnh xám có kích thước `28 × 28`; nhãn là một số nguyên từ 0 đến 9. Test được giữ riêng để đánh giá cuối cùng, không dùng để lựa chọn mô hình.

In [ ]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = (
    tf.keras.datasets.mnist.load_data()
)

print("Ảnh train:", x_train_raw.shape, x_train_raw.dtype)
print("Nhãn train:", y_train_raw.shape, y_train_raw.dtype)
print("Ảnh test: ", x_test_raw.shape, x_test_raw.dtype)
print("Nhãn test: ", y_test_raw.shape, y_test_raw.dtype)
print("Miền pixel train:", int(x_train_raw.min()), "đến", int(x_train_raw.max()))
print("Các nhãn:", np.unique(y_train_raw))

## 4. Xem một ảnh và nhãn tương ứng

Một phần tử `x_train_raw[i]` là ma trận 28 × 28. Giá trị pixel ban đầu là số nguyên trong `[0, 255]`; `y_train_raw[i]` là đáp án đúng của chính ảnh đó. `cmap="gray"` hiển thị ảnh theo thang xám.

In [ ]:
sample_index = 0
plt.figure(figsize=(3, 3))
plt.imshow(x_train_raw[sample_index], cmap="gray")
plt.title(f"Nhãn thật: {y_train_raw[sample_index]}")
plt.axis("off")
plt.show()

## 5. Vì sao đây là phân loại đa lớp?

Mỗi ảnh thuộc đúng một trong 10 lớp chữ số, nên đây là **phân loại đa lớp** (không phải hồi quy hay phân loại nhị phân). Lớp cuối có 10 neuron, một neuron cho mỗi chữ số. Softmax chuyển 10 giá trị đầu ra thành phân phối xác suất: mỗi giá trị nằm trong `[0, 1]` và tổng gần bằng 1.

Nhãn gốc là một số nguyên, ví dụ `3`. Vì ta chọn `categorical_crossentropy`, cần đổi nhãn thành one-hot: `3` thành `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`. Nếu muốn giữ nhãn số nguyên, có thể dùng `sparse_categorical_crossentropy` thay thế và bỏ bước one-hot.

In [ ]:
NUM_CLASSES = 10
y_train_one_hot = to_categorical(y_train_raw, num_classes=NUM_CLASSES)
y_test_one_hot = to_categorical(y_test_raw, num_classes=NUM_CLASSES)

print("Nhãn gốc:", y_train_raw[0])
print("Nhãn one-hot:", y_train_one_hot[0])
print("Shape nhãn train:", y_train_one_hot.shape)
print("Shape nhãn test: ", y_test_one_hot.shape)

## 6. Tiền xử lý ảnh

Chia pixel cho 255 đưa chúng từ `[0, 255]` về `[0, 1]`, giúp tối ưu hóa ổn định hơn. Ảnh gốc có shape `(N, 28, 28)`. `Conv2D` mặc định nhận dữ liệu dạng `(N, chiều_cao, chiều_rộng, số_kênh)`, vì vậy ta thêm một kênh xám: `(N, 28, 28, 1)`. Ảnh RGB sẽ có 3 kênh thay vì 1.

In [ ]:
x_train = x_train_raw.astype("float32") / 255.0
x_test = x_test_raw.astype("float32") / 255.0

x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print("Shape ảnh train sau xử lý:", x_train.shape)
print("Shape ảnh test sau xử lý: ", x_test.shape)
print("Miền pixel sau xử lý:", float(x_train.min()), "đến", float(x_train.max()))

## 7. Tách tập validation

Validation dùng trong quá trình phát triển để theo dõi khả năng tổng quát hóa và quyết định dừng huấn luyện. Ta trích 5.000 ảnh từ tập train, có phân tầng theo nhãn (`stratify`) để tỷ lệ các chữ số tương tự giữa hai phần. **Tập test không tham gia bước này**; chỉ dùng một lần ở phần đánh giá sau huấn luyện.

In [ ]:
x_train_fit, x_val, y_train_fit, y_val = train_test_split(
    x_train,
    y_train_one_hot,
    test_size=5000,
    random_state=SEED,
    stratify=y_train_raw,
)

print("Train dùng để fit:", x_train_fit.shape, y_train_fit.shape)
print("Validation:       ", x_val.shape, y_val.shape)

## 8. Xây dựng CNN

- **Conv2D:** 32 bộ lọc kích thước 4 × 4, `valid` nghĩa là không thêm padding; mỗi bộ lọc học một kiểu đặc trưng cục bộ như nét/cạnh.
- **ReLU:** tạo phi tuyến, giữ giá trị dương và đưa giá trị âm về 0.
- **MaxPooling2D:** lấy giá trị lớn nhất trong từng vùng 2 × 2, giảm kích thước không gian mà vẫn giữ số kênh đặc trưng.
- **Flatten:** trải các bản đồ đặc trưng thành vector để đưa vào lớp fully connected.
- **Dense(128):** kết hợp các đặc trưng đã trích xuất.
- **Dense(10, softmax):** tạo xác suất cho 10 chữ số.

Shape đi qua mạng (không tính batch): `28×28×1 → 25×25×32 → 12×12×32 → 4608 → 128 → 10`. Conv hợp lệ cho kích thước `28 - 4 + 1 = 25`. Pooling 2×2, stride mặc định 2, cho `floor((25 - 2)/2) + 1 = 12`.

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(
            filters=32,
            kernel_size=(4, 4),
            padding="valid",
            activation="relu",
            name="conv_relu",
        ),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name="max_pool"),
        tf.keras.layers.Flatten(name="flatten"),
        tf.keras.layers.Dense(128, activation="relu", name="dense_hidden"),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="class_probabilities"),
    ],
    name="mnist_cnn",
)

model.summary()

### 8.1. Đọc số tham số

Tham số gồm trọng số và bias có thể học; lớp pooling và flatten không có tham số học được.

- Conv2D: `(4 × 4 × 1 + 1) × 32 = 544` (mỗi bộ lọc có 16 trọng số và 1 bias).
- Dense 128: `(12 × 12 × 32) × 128 + 128 = 589.952`.
- Dense 10: `128 × 10 + 10 = 1.290`.
- Tổng: `544 + 589.952 + 1.290 = 591.786` tham số.

`model.summary()` là cách kiểm tra shape và số tham số do Keras tính. Nên tự tính lại để hiểu kết nối giữa các lớp, không chỉ ghi nhớ tổng số.

In [ ]:
expected_parameters = 544 + (12 * 12 * 32 * 128 + 128) + (128 * 10 + 10)
print("Số tham số theo phép tính:", expected_parameters)
print("Số tham số trong model:   ", model.count_params())
assert model.count_params() == expected_parameters

## 9. Cấu hình quá trình học

`categorical_crossentropy` đo mức khác biệt giữa phân phối xác suất dự đoán và nhãn one-hot đúng. Adam tự điều chỉnh bước cập nhật tham số trong quá trình tối ưu. `accuracy` là tỷ lệ mẫu có lớp dự đoán đúng; đây là chỉ số dễ diễn giải, nhưng chưa thể hiện đầy đủ lỗi của từng lớp.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

## 10. Huấn luyện và EarlyStopping

Một **epoch** là một lượt mô hình đi qua toàn bộ dữ liệu train dùng để fit. Với `batch_size=128`, mô hình cập nhật sau từng nhóm tối đa 128 ảnh. Ta đặt tối đa 10 epoch, nhưng `EarlyStopping` dừng nếu `val_loss` không cải thiện trong 2 epoch liên tiếp. `restore_best_weights=True` khôi phục trọng số ở epoch validation tốt nhất.

Không nên khẳng định trước mô hình sẽ dừng ở epoch nào hoặc đạt accuracy cụ thể: kết quả phụ thuộc môi trường và quá trình chạy.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    x_train_fit,
    y_train_fit,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1,
)

## 11. Đọc đồ thị loss và accuracy

Đường train mô tả hiệu quả trên dữ liệu mà mô hình đang học; đường validation mô tả hiệu quả trên dữ liệu được giữ riêng. Loss train tiếp tục giảm trong khi loss validation tăng kéo dài có thể là dấu hiệu **overfitting**. EarlyStopping theo dõi `val_loss`, không dựa trên tập test.

In [ ]:
epochs_ran = range(1, len(history.history["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_ran, history.history["loss"], label="Train")
axes[0].plot(epochs_ran, history.history["val_loss"], label="Validation")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Categorical cross-entropy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history.history["accuracy"], label="Train")
axes[1].plot(epochs_ran, history.history["val_accuracy"], label="Validation")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Đánh giá trên tập test

Sau khi đã hoàn tất lựa chọn/huấn luyện mô hình bằng train và validation, ta đánh giá trên test. `evaluate` trả về loss và accuracy trên 10.000 ảnh test. Đây là ước lượng hiệu quả trên dữ liệu chưa dùng trong quá trình học; không nên liên tục tinh chỉnh mô hình theo test vì sẽ làm mất tính khách quan của tập này.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test_one_hot, verbose=0)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

## 13. Từ xác suất sang nhãn dự đoán

`model.predict` trả về một vector 10 xác suất cho mỗi ảnh. `argmax(axis=1)` lấy chỉ số có xác suất cao nhất, chính là lớp dự đoán. Ta so sánh lớp dự đoán với nhãn số nguyên gốc để xem precision, recall, F1-score theo từng chữ số.

In [ ]:
test_probabilities = model.predict(x_test, batch_size=256, verbose=0)
y_test_pred = np.argmax(test_probabilities, axis=1)

print(classification_report(y_test_raw, y_test_pred, digits=4, zero_division=0))

### 13.1. Confusion matrix

Hàng là nhãn thật, cột là nhãn dự đoán. Các giá trị trên đường chéo là dự đoán đúng; ngoài đường chéo là nhầm lẫn giữa hai chữ số. Ma trận giúp tìm kiểu lỗi mà một con số accuracy tổng thể không thể hiện.

In [ ]:
matrix = confusion_matrix(y_test_raw, y_test_pred, labels=np.arange(NUM_CLASSES))
display = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=np.arange(NUM_CLASSES),
)
display.plot(cmap="Blues", values_format="d", colorbar=False)
plt.title("Confusion matrix trên tập test")
plt.show()

## 14. Thử dự đoán một ảnh

Ảnh mẫu bên dưới lấy từ `x_test`, đã được chia 255 và có shape `(28, 28, 1)`. Khi chuẩn bị **ảnh bên ngoài**, cần áp dụng cùng quy trình tiền xử lý: ảnh xám 28 × 28, cùng hướng/nền tương thích, kiểu số thực, chia 255 và thêm chiều batch cùng chiều kênh để có shape `(1, 28, 28, 1)`. Không chuẩn hóa lại ảnh vốn đã ở `[0,1]`.

In [ ]:
inference_index = 0
inference_image = x_test[inference_index]
inference_batch = np.expand_dims(inference_image, axis=0)

probabilities = model.predict(inference_batch, verbose=0)[0]
predicted_digit = int(np.argmax(probabilities))
true_digit = int(y_test_raw[inference_index])

plt.figure(figsize=(3, 3))
plt.imshow(inference_image.squeeze(), cmap="gray")
plt.title(f"Thật: {true_digit} | Dự đoán: {predicted_digit}")
plt.axis("off")
plt.show()

print("Shape batch đầu vào:", inference_batch.shape)
print("Xác suất theo lớp 0..9:", np.round(probabilities, 3))

## 15. Tổng kết và các lỗi thường gặp

- MNIST là bài toán phân loại 10 lớp; lớp cuối có 10 đầu ra softmax.
- Dùng `categorical_crossentropy` thì nhãn phải one-hot; với nhãn nguyên, dùng `sparse_categorical_crossentropy`.
- CNN nhận ảnh theo dạng `(batch, height, width, channels)`; ảnh xám có 1 kênh.
- Chuẩn hóa train, validation, test và ảnh inference theo cùng một cách.
- Validation hỗ trợ theo dõi quá trình học; test chỉ dùng để đánh giá cuối cùng.
- `argmax` chuyển vector xác suất thành một lớp dự đoán.
- Mô hình được xây bằng Keras; phần tích chập và backpropagation không được tự cài đặt trong notebook này.

**Bài tập mở rộng:** thử thay kernel 4 × 4 thành 3 × 3, thêm một khối Conv2D + MaxPooling2D, hoặc so sánh với mô hình dùng `sparse_categorical_crossentropy`. Sau mỗi thay đổi, kiểm tra lại shape, số tham số và đồ thị validation.